In [ ]:
import itertools
import logging
import os
import pickle
import random
import sys
from os.path import join

import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from Bio import SeqIO
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/Screening_and_application


## 1.read

In [ ]:
! python distinct.py --input "/home/hanxd/Repositories/ESP/data/our_data/screening1/Zea_mays_pdb_PF00067_div_right.pep" --output "/home/hanxd/Repositories/ESP/data/our_data/screening1/Zea_mays_pdb_PF00067_div_right_q.fasta"
! python distinct.py --input "/home/hanxd/Repositories/ESP/data/our_data/screening1/Glycine_max_pdb_PF00067_div_right.pep" --output "/home/hanxd/Repositories/ESP/data/our_data/screening1/Glycine_max_pdb_PF00067_div_right_q.fasta"
! python distinct.py --input "/home/hanxd/Repositories/ESP/data/our_data/screening1/Erigeron_breviscapus_PF00067_div_right.pep" --output "/home/hanxd/Repositories/ESP/data/our_data/screening1/Erigeron_breviscapus_PF00067_div_right_q.fasta"

In [ ]:
deletedata = pd.read_pickle(our_data + "screening1/" + "deletedata.pkl")

In [5]:
len(deletedata)

29

In [ ]:
zea_enzyme = [
    "CYP701A26",
    "CYP701A43",
    "CYP706C37",
    "CYP711A3Zm",
    "CYP71C1",
    "CYP71C2",
    "CYP71C3",
    "CYP71C4",
    "CYP71Z16",
    "CYP71Z18",
    "CYP71Z19",
    "CYP78A1",
    "CYP79A61",
    "CYP81A37",
    "CYP81A38",
    "CYP81A39",
    "CYP92C5",
    "CYP92C6",
]
Glycine_enzyme = [
    "CYP71A10",
    "CYP71D9",
    "CYP72A69",
    "CYP81E100",
    "CYP82D26",
    "CYP93A1",
    "CYP93B16",
    "CYP93C1",
    "CYP93E1",
]
Erigeron_enzyme = ["CYP706X1", "CYP73A500"]

In [ ]:
zea_deletedata = deletedata[deletedata["enzyme"].isin(zea_enzyme)]
glycine_deletedata = deletedata[deletedata["enzyme"].isin(Glycine_enzyme)]
erigeron_deletedata = deletedata[deletedata["enzyme"].isin(Erigeron_enzyme)]

zea_deletedata.to_pickle(our_data + "screening1/" + "zea_deletedata.pkl")
glycine_deletedata.to_pickle(our_data + "screening1/" + "glycine_deletedata.pkl")
erigeron_deletedata.to_pickle(our_data + "screening1/" + "erigeron_deletedata.pkl")

In [ ]:
erisublist = erigeron_deletedata["substrate"].unique().tolist()
glysublist = glycine_deletedata["substrate"].unique().tolist()
zeasublist = zea_deletedata["substrate"].unique().tolist()

print(len(zeasublist))
print(len(glysublist))
print(len(erisublist))

13
7
2


In [8]:
glysublist

['SOY', 'NAR', 'fluometuron', 'LIQ', 'DIT', 'BAM', 'DAI']

In [ ]:
enzyme_list = []
sequence_list = []
substrate_list = []
for record in SeqIO.parse(
    "/home/hanxd/Repositories/ESP/data/our_data/screening1/Zea_mays_pdb_PF00067_div_right_q.fasta",
    "fasta",
):
    for i in zeasublist:
        substrate_list.append(i)
        enzyme_list.append(record.id)
        sequence_list.append(str(record.seq))
dfzea = pd.DataFrame(
    {"enzyme": enzyme_list, "sequence": sequence_list, "substrate": substrate_list}
)


enzyme_list = []
sequence_list = []
substrate_list = []
for record in SeqIO.parse(
    "/home/hanxd/Repositories/ESP/data/our_data/screening1/Glycine_max_pdb_PF00067_div_right_q.fasta",
    "fasta",
):
    for i in glysublist:
        substrate_list.append(i)
        enzyme_list.append(record.id)
        sequence_list.append(str(record.seq))

dfgly = pd.DataFrame(
    {"enzyme": enzyme_list, "sequence": sequence_list, "substrate": substrate_list}
)

enzyme_list = []
sequence_list = []
substrate_list = []
for record in SeqIO.parse(
    "/home/hanxd/Repositories/ESP/data/our_data/screening1/Erigeron_breviscapus_PF00067_div_right_q.fasta",
    "fasta",
):
    for i in erisublist:
        substrate_list.append(i)
        enzyme_list.append(record.id)
        sequence_list.append(str(record.seq))
dferi = pd.DataFrame(
    {"enzyme": enzyme_list, "sequence": sequence_list, "substrate": substrate_list}
)

In [10]:
dferi

,enzyme,sequence,substrate
0,Ebre_g024454,MEIPYIYISLIILLSCYLFTTIFRRKTTNLPPTIFPSLPIIGHLYL...,AGI
1,Ebre_g024454,MEIPYIYISLIILLSCYLFTTIFRRKTTNLPPTIFPSLPIIGHLYL...,CIN
2,Ebre_g062151,METFGRVAVTIFIVVIVRWGWKVVNWVWLKPKKLEKWLRNEGYKGN...,AGI
3,Ebre_g062151,METFGRVAVTIFIVVIVRWGWKVVNWVWLKPKKLEKWLRNEGYKGN...,CIN
4,Ebre_g049202,MDTLIFPPLWLLSTVVIIFMSFIFAYTRKSNKSSAKLPPSPPRLPI...,AGI
...,...,...,...
407,Ebre_g064963,MELQIHFFALLIFCIAFFYLFTKRRSSSGNENLPPQPWKLPLLGHL...,CIN
408,Ebre_g043614,MLSYLITISALILFFVSLLSKFRTPTTKTLRPLPPGPPGLPIIGNL...,AGI
409,Ebre_g043614,MLSYLITISALILFFVSLLSKFRTPTTKTLRPLPPGPPGLPIIGNL...,CIN
410,Viridiplantae_CYP706X1,MASNELAFSALLVTLVLVLISWYKREISNSRKAGTPPLPPGPKGLP...,AGI


## 2.generate

In [11]:
# ! python extract.py esm1b_t33_650M_UR50S {our_data}/screening1/Zea_mays_pdb_PF00067_div_right_q.fasta {our_data}/screening1/esm --repr_layers 33 --include mean
# ! python extract.py esm1b_t33_650M_UR50S {our_data}/screening1/Glycine_max_pdb_PF00067_div_right_q.fasta {our_data}/screening1/esm --repr_layers 33 --include mean
# ! python extract.py esm1b_t33_650M_UR50S {our_data}/screening1/Erigeron_breviscapus_PF00067_div_right_q.fasta {our_data}/screening1/esm --repr_layers 33 --include mean

Transferred model to GPU
Read /home/hanxd/Repositories/ESP/Screening_and_application/../data/our_data/screening1/Zea_mays_pdb_PF00067_div_right_q.fasta with 229 sequences
Processing 1 of 32 batches (9 sequences)
Processing 2 of 32 batches (8 sequences)
Processing 3 of 32 batches (8 sequences)
Processing 4 of 32 batches (8 sequences)
Processing 5 of 32 batches (8 sequences)
Processing 6 of 32 batches (8 sequences)
Processing 7 of 32 batches (8 sequences)
Processing 8 of 32 batches (8 sequences)
Processing 9 of 32 batches (8 sequences)
Processing 10 of 32 batches (8 sequences)
Processing 11 of 32 batches (7 sequences)
Processing 12 of 32 batches (7 sequences)
Processing 13 of 32 batches (7 sequences)
Processing 14 of 32 batches (7 sequences)
Processing 15 of 32 batches (7 sequences)
Processing 16 of 32 batches (7 sequences)
Processing 17 of 32 batches (7 sequences)
Processing 18 of 32 batches (7 sequences)
Processing 19 of 32 batches (7 sequences)
Processing 20 of 32 batches (7 sequences

In [ ]:
dfzea["ESM1b"] = ""
dfzea["ECFP"] = ""

rep_dict = join(CURRENT_DIR, "..", "data", "our_data", "screening1/esm/")

for ind in dfzea.index:
    esms = torch.load(rep_dict + str(dfzea["enzyme"][ind]) + ".pt")
    sdf_file_path = our_data + "screening1/SDF_300/" + dfzea["substrate"][ind] + ".sdf"
    mol = Chem.MolFromMolFile(sdf_file_path)
    ecfpso = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024).ToBitString()
    dfzea["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    dfzea["ECFP"][ind] = ecfpso
dfzea["Binding"] = 0

In [ ]:
dfzea.to_pickle(our_data + "screening1/" + "dfzea.pkl")

In [ ]:
dfzea["substrate"].nunique()

13

In [ ]:
dfgly["ESM1b"] = ""
dfgly["ECFP"] = ""

rep_dict = join(CURRENT_DIR, "..", "data", "our_data", "screening1/esm/")

for ind in dfgly.index:
    esms = torch.load(rep_dict + str(dfgly["enzyme"][ind]) + ".pt")
    sdf_file_path = our_data + "screening1/SDF_300/" + dfgly["substrate"][ind] + ".sdf"
    mol = Chem.MolFromMolFile(sdf_file_path)
    ecfpso = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024).ToBitString()
    dfgly["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    dfgly["ECFP"][ind] = ecfpso
dfgly["Binding"] = 0

In [ ]:
dfgly.to_pickle(our_data + "screening1/" + "dfgly.pkl")

In [ ]:
dfgly["substrate"].nunique()

7

In [ ]:
dferi["ESM1b"] = ""
dferi["ECFP"] = ""

rep_dict = join(CURRENT_DIR, "..", "data", "our_data", "screening1/esm/")

for ind in dferi.index:
    esms = torch.load(rep_dict + str(dferi["enzyme"][ind]) + ".pt")
    sdf_file_path = our_data + "screening1/SDF_300/" + dferi["substrate"][ind] + ".sdf"
    mol = Chem.MolFromMolFile(sdf_file_path)
    ecfpso = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024).ToBitString()
    dferi["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    dferi["ECFP"][ind] = ecfpso
dferi["Binding"] = 0

In [ ]:
dferi.to_pickle(our_data + "screening1/" + "dferi.pkl")

In [ ]:
dferi["substrate"].nunique()

2

In [21]:
dferi

,enzyme,sequence,substrate,ESM1b,ECFP,Binding
0,Ebre_g024454,MEIPYIYISLIILLSCYLFTTIFRRKTTNLPPTIFPSLPIIGHLYL...,AGI,"[-0.029463786631822586, 0.08772501349449158, -...",0000000000000000000000000000000000000000000000...,0
1,Ebre_g024454,MEIPYIYISLIILLSCYLFTTIFRRKTTNLPPTIFPSLPIIGHLYL...,CIN,"[-0.029463786631822586, 0.08772501349449158, -...",0000000000000000000000000000000100000000000000...,0
2,Ebre_g062151,METFGRVAVTIFIVVIVRWGWKVVNWVWLKPKKLEKWLRNEGYKGN...,AGI,"[-0.08078508824110031, 0.1934240311384201, -0....",0000000000000000000000000000000000000000000000...,0
3,Ebre_g062151,METFGRVAVTIFIVVIVRWGWKVVNWVWLKPKKLEKWLRNEGYKGN...,CIN,"[-0.08078508824110031, 0.1934240311384201, -0....",0000000000000000000000000000000100000000000000...,0
4,Ebre_g049202,MDTLIFPPLWLLSTVVIIFMSFIFAYTRKSNKSSAKLPPSPPRLPI...,AGI,"[-0.048562806099653244, 0.184627965092659, -0....",0000000000000000000000000000000000000000000000...,0
...,...,...,...,...,...,...
407,Ebre_g064963,MELQIHFFALLIFCIAFFYLFTKRRSSSGNENLPPQPWKLPLLGHL...,CIN,"[-0.07572026550769806, 0.14106214046478271, -0...",0000000000000000000000000000000100000000000000...,0
408,Ebre_g043614,MLSYLITISALILFFVSLLSKFRTPTTKTLRPLPPGPPGLPIIGNL...,AGI,"[-0.060216017067432404, 0.13754795491695404, -...",0000000000000000000000000000000000000000000000...,0
409,Ebre_g043614,MLSYLITISALILFFVSLLSKFRTPTTKTLRPLPPGPPGLPIIGNL...,CIN,"[-0.060216017067432404, 0.13754795491695404, -...",0000000000000000000000000000000100000000000000...,0
410,Viridiplantae_CYP706X1,MASNELAFSALLVTLVLVLISWYKREISNSRKAGTPPLPPGPKGLP...,AGI,"[-0.007465010974556208, 0.1468241959810257, 0....",0000000000000000000000000000000000000000000000...,0
